# SismoReporta AQP - Generación de Vista de Despliegue (E6)

Este notebook de Google Colab permite generar automáticamente el diagrama de despliegue requerido en la práctica de laboratorio **E6**, utilizando la biblioteca `diagrams` y `Graphviz`.

In [ ]:
# 1. Instalación de Graphviz a nivel de sistema y la librería diagrams de Python
!apt-get update -qq
!apt-get install -y -qq graphviz
!pip install -q diagrams

In [ ]:
# 2. Código del diagrama de despliegue
import os
from diagrams import Diagram, Cluster, Edge
from diagrams.generic.device import Mobile
from diagrams.onprem.client import User
from diagrams.onprem.network import Nginx, Internet
from diagrams.programming.framework import Fastapi
from diagrams.onprem.inmemory import Redis
from diagrams.onprem.database import PostgreSQL
from diagrams.generic.storage import Storage
from diagrams.onprem.monitoring import Prometheus, Grafana

os.makedirs("img", exist_ok=True)

graph_attr = {
    "fontsize": "16",
    "fontname": "Helvetica",
    "bgcolor": "#F8F9FA",
    "pad": "0.5",
    "splines": "ortho"
}

with Diagram(
    "SismoReporta AQP - Vista de Despliegue Física y Lógica",
    filename="img/despliegue",
    outformat="png",
    show=False,
    direction="TB",
    graph_attr=graph_attr
):
    with Cluster("Dispositivos de Clientes y Actores"):
        with Cluster("Dispositivos Móviles (Offline-First)"):
            app_ciudadano = Mobile("App Móvil Ciudadano\n(Android / iOS)\n[SQLite Local + Sync]")
        with Cluster("Puesto de Comando"):
            operador = User("Operador / Defensa Civil\n[Navegador Web / PC]")

    red_movil = Internet("Red Móvil Celular\n(Intermitente / Post-sismo)")
    red_internet = Internet("Red Internet / WAN")

    with Cluster("Servidor de Producción SismoReporta AQP (Host / Docker)"):
        proxy = Nginx("Nginx Reverse Proxy\n[SSL Termination & Rate Limit]")

        with Cluster("Contenedor Backend (Monolito Modular)"):
            backend_api = Fastapi("API REST (FastAPI)\n- Mód. Ingesta (UUID)\n- Mód. Operaciones")
            worker_interno = Fastapi("Worker Asíncrono Interno\n[Consumidor de Cola]")

        with Cluster("Caché y Cola Persistente"):
            cola_redis = Redis("Redis 7 (AOF Persistente)\n[Cola BullMQ/Celery >20k msgs]")

        with Cluster("Almacenamiento y Base de Datos"):
            bd_postgres = PostgreSQL("PostgreSQL 16 + PostGIS\n[Reportes, Zonas, Brigadas]")
            storage_fotos = Storage("Storage de Evidencia (MinIO)\n[Fotos / Vol. Persistente]")

        with Cluster("Observabilidad y Monitoreo"):
            prom = Prometheus("Prometheus\n[Métricas de Cola y Latencia]")
            grafana = Grafana("Grafana Dashboard\n[Alertas y Rendimiento]")

    with Cluster("Servicios Externos"):
        mapas_externos = Internet("Servicio Externo de Mapas\n(OpenStreetMap / Mapbox Tiles)")

    # Conexiones
    app_ciudadano >> Edge(label="1. Reporte offline o reintento\n(HTTPS / JSON + UUID)", color="#D9534F") >> red_movil
    red_movil >> Edge(label="HTTPS / 443", color="#D9534F") >> proxy
    operador >> Edge(label="Consulta panel / Asignar brigada\n(HTTPS / REST)", color="#0275D8") >> red_internet
    red_internet >> Edge(label="HTTPS / 443", color="#0275D8") >> proxy

    proxy >> Edge(label="Proxy pass :8000", color="#333333") >> backend_api
    backend_api >> Edge(label="2. Encolar payload liviano\n[Retorna HTTP 202 en <100ms]", color="#5CB85C") >> cola_redis
    cola_redis >> Edge(label="3. Pop / Consumo ordenado", color="#F0AD4E") >> worker_interno
    worker_interno >> Edge(label="4. INSERT reporte (PostGIS)", color="#2E6DA4") >> bd_postgres
    worker_interno >> Edge(label="5. Guardar fotos referenciadas", color="#5BC0DE") >> storage_fotos
    backend_api >> Edge(label="Consultar / Priorizar zonas", color="#0275D8") >> bd_postgres
    operador >> Edge(label="Descarga de Map Tiles", style="dotted", color="#8A6D3B") >> mapas_externos
    backend_api >> Edge(label="/metrics", style="dashed", color="#777777") >> prom
    cola_redis >> Edge(label="Métricas cola", style="dashed", color="#777777") >> prom
    prom >> Edge(label="Visualizar", color="#F39C12") >> grafana

print("¡Diagrama generado exitosamente en img/despliegue.png!")

In [ ]:
# 3. Mostrar la imagen generada en el notebook
from IPython.display import Image, display
display(Image("img/despliegue.png"))